In [49]:
import pandas as pd
import torch

In [50]:
#get the data

df = pd.read_csv('../data/era_adjusted_pitching.csv')
df = df.sample(frac=1, random_state=42).reset_index(drop=True)
df.head()

,fullName,era_plus,dice_plus,k_pct_plus,bb_pct_plus,hr9_plus
0,Carlos Zambrano,102.450890,95.630028,99.757613,75.273082,124.833611
1,Vance Worley,118.278071,109.635023,100.367468,110.138486,126.285817
2,Casey Fossum,93.933440,95.705479,114.968477,90.868781,89.347378
3,Mike Smithson,84.486277,87.372614,86.027793,132.806524,70.733474
4,Jose Berrios,103.783579,97.656076,98.869186,111.196329,93.708863


In [51]:
features = torch.tensor(df[['dice_plus', 'k_pct_plus', 'bb_pct_plus', 'hr9_plus']].values, dtype=torch.float32)
target = torch.tensor(df[['era_plus']].values, dtype=torch.float32)

print(features.shape)
print(target.shape)

torch.Size([806, 4])
torch.Size([806, 1])


In [52]:


X_train, X_test = features[:643], features[643:]
Y_train, Y_test = target[:643], target[643:]


X_test.shape

torch.Size([163, 4])

In [53]:
import torch.nn as nn

class PitcherModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(4, 16),  # 4 features in, 16 neurons
            nn.ReLU(),
            nn.Linear(16, 1)   # 16 neurons in, 1 output (ERA+)
        )
    
    def forward(self, x):
        return self.network(x)

model = PitcherModel()
print(model)

PitcherModel(
  (network): Sequential(
    (0): Linear(in_features=4, out_features=16, bias=True)
    (1): ReLU()
    (2): Linear(in_features=16, out_features=1, bias=True)
  )
)


In [54]:
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

In [55]:
epochs = 1000

for epoch in range(epochs):
    # forward pass
    predictions = model(X_train)
    
    # calculate loss
    loss = loss_fn(predictions, Y_train)
    
    # backward pass
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    # print progress every 10 epochs
    if epoch % 10 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")

Epoch 0, Loss: 8543.6543
Epoch 10, Loss: 301.5682
Epoch 20, Loss: 447.8636
Epoch 30, Loss: 236.7569
Epoch 40, Loss: 90.0244
Epoch 50, Loss: 100.0071
Epoch 60, Loss: 96.8807
Epoch 70, Loss: 89.7252
Epoch 80, Loss: 87.6252
Epoch 90, Loss: 87.2013
Epoch 100, Loss: 86.9075
Epoch 110, Loss: 86.5919
Epoch 120, Loss: 86.2826
Epoch 130, Loss: 86.0185
Epoch 140, Loss: 85.7923
Epoch 150, Loss: 85.5874
Epoch 160, Loss: 85.3981
Epoch 170, Loss: 85.2313
Epoch 180, Loss: 85.0716
Epoch 190, Loss: 84.9187
Epoch 200, Loss: 84.7759
Epoch 210, Loss: 84.6423
Epoch 220, Loss: 84.5176
Epoch 230, Loss: 84.3966
Epoch 240, Loss: 84.2822
Epoch 250, Loss: 84.1735
Epoch 260, Loss: 84.0638
Epoch 270, Loss: 83.9618
Epoch 280, Loss: 83.8609
Epoch 290, Loss: 83.7648
Epoch 300, Loss: 83.6803
Epoch 310, Loss: 83.6036
Epoch 320, Loss: 83.5324
Epoch 330, Loss: 83.4632
Epoch 340, Loss: 83.3941
Epoch 350, Loss: 83.3256
Epoch 360, Loss: 83.2588
Epoch 370, Loss: 83.1941
Epoch 380, Loss: 83.1289
Epoch 390, Loss: 83.0672
Epoch

In [56]:
model.eval()
with torch.no_grad():
    test_predictions = model(X_test)
    test_loss = loss_fn(test_predictions, Y_test)
    print(f"Test Loss: {test_loss.item():.4f}")

Test Loss: 91.8536


In [57]:
# get your original dataframe index for test pitchers
test_df = df.iloc[643:].copy()
test_df['predicted_era_plus'] = test_predictions.detach().numpy()
test_df['error'] = test_df['predicted_era_plus'] - test_df['era_plus']

print(test_df[['fullName', 'era_plus', 'predicted_era_plus', 'error']].to_string())

test_df["x100"] = test_df.apply(lambda x: 1 if x['era_plus'])

SyntaxError: expected 'else' after 'if' expression (3043887948.py, line 8)

## Notes

- Started by getting all the era adjusted data for current pitchers
- created feature and target tensors. feature are going to be used to make the inference, target is era+ which is what we're going to predict
- split into test and train groups